# AIR Homework 1 (Task 1)

## Setup

This notebook covers Task 1. If you haven't already, read `start_here.ipynb` first. It has the assignment overview, the submission instructions, and how to set up the conda environment.

Run the cells below before you start. They set up the notebook, load our working environment from `scene.py`, and import your code from `task1.py`.

In [ ]:
%load_ext autoreload
%autoreload 2

# set plot theme
import seaborn as sns
sns.set_theme()

# Set the EGL backend for MuJoCo on Linux (no effect on other operating systems)
import platform

if platform.system() == "Linux":
  %env MUJOCO_GL=egl

In [ ]:
# load our working environment from scene.py (see start_here.ipynb)
from scene import data, model

import mujoco
import mediapy as media
import numpy as np

In [ ]:
# import your code for this task. the rest of this notebook calls it through the
# module, e.g. t1.get_ee_pose
import task1 as t1

## Task 1: Mapping Between Spaces

In the previous tutorial, you implemented a proportional controller to control the end-effector position. There are two reasons why this isn't sufficient for our pick-and-place task:
1. We are going to use motion planning to navigate around obstacles, which means we will be working in configuration space rather than end-effector space.
2. We need to control both the position and orientation (A.K.A. **pose**) of the end-effector to successfully grasp

Therefore, we will be implementing our controller in configuration space. However, to get meaningful configurations, we must first implement an efficient method to translate between end-effector poses and robot configurations.

### Forward and Inverse Kinematics

We would like to use the UR5e robot to manipulate objects in its workspace. For this, we need to place the end effector at specific coordinates and orientation within the workspace, i.e., the physical space where the robot interacts with objects. This task requires converting between the workspace and the C-space.

![forward and inverse kinematic](https://github.com/CLAIR-LAB-TECHNION/ICAPS-24/blob/main/assets/figures/f_inv_kinematics.png?raw=true)

The process of mapping from a robot's configuration—specifically its joint angles—to the workspace—where the end effector's position and orientation are defined—is known as **forward kinematics**. This mapping is indeed feasible and relatively intuitive if we know the robot's geometric parameters, such as the lengths and relative positions of its links. The computation of the end effector's pose from the joint configurations is typically represented using matrix multiplications. These matrices, often referred to as transformation matrices, sequentially transform the pose from one joint to the next along the robot's arm, ultimately resulting in the pose of the end effector in the workspace.

As our tutorial on proportional control, we will bypass implementing forward kinematics for the UR5e robot using by getting the end effector's pose directly from the simulation environment. Make sure to return a copy of the pose arrays since these update in-place in the simulation. Also, remember that all robot elements were attached to the scene with the prefix "robot_".

**TODO:** implement `get_ee_pose` in `task1.py`.

Let's make sure this works. From here on, cells that start with `# sanity check` run a few quick checks on what you just implemented, and print what they found. If one fails, something is wrong. If it passes, you are heading in the right direction, but your code may still have bugs: these checks are not the tests we will run on your submission.

In [ ]:
import unittest

test = unittest.TestCase()


def get_check_data():
    """Get a separate simulation state at the home configuration, so that the sanity
    checks don't affect the rest of this notebook."""
    check_data = mujoco.MjData(model)
    mujoco.mj_resetDataKeyframe(model, check_data, model.key("robot_home").id)
    mujoco.mj_forward(model, check_data)
    return check_data

In [ ]:
# sanity check: get_ee_pose
home_ee_pos = [0.492, 0.134, 1.208]
home_ee_rot_mat = [[0, -1, 0], [-1, 0, 0], [0, 0, -1]]

# the pose at the home configuration
check_data = get_check_data()
ee_pos, ee_rot_mat = t1.get_ee_pose(check_data)
np.testing.assert_allclose(ee_pos, home_ee_pos, atol=1e-3)
np.testing.assert_allclose(ee_rot_mat, home_ee_rot_mat, atol=1e-3)

# the pose at a different configuration
mujoco.mj_resetData(model, check_data)
mujoco.mj_forward(model, check_data)
new_ee_pos, new_ee_rot_mat = t1.get_ee_pose(check_data)
np.testing.assert_allclose(new_ee_pos, [-0.234, 0.817, 0.783], atol=1e-3)
np.testing.assert_allclose(new_ee_rot_mat, [[0, 0, -1], [-1, 0, 0], [0, 1, 0]], atol=1e-3)

# the first pose is a copy, so moving the robot didn't change it
np.testing.assert_allclose(ee_pos, home_ee_pos, atol=1e-3)
np.testing.assert_allclose(ee_rot_mat, home_ee_rot_mat, atol=1e-3)

print(f"get_ee_pose sanity checks passed. position at home = {np.round(ee_pos, 3)}")

**Inverse kinematics (IK)** involves determining the joint configurations that will achieve a specified position and orientation of the end effector in the workspace. This process is the reverse of forward kinematics and can be more complex because there may be multiple, or even an infinite number of, joint configurations that can result in the same end effector pose. IK essentially frames this challenge as an optimization problem, where the goal is to find the best set of joint angles that satisfy the end effector's target position and orientation. Unlike the direct calculations used in forward kinematics, solving inverse kinematics often requires the use of numerical methods and optimization techniques to find feasible solutions. However, there are analytical solutions for specific robots, including the UR5e, which can efficiently calculate exact joint angles for given end effector poses.

### Reference Frame Transformations

![reference frames](https://figures.semanticscholar.org/c6d14b7a87d664d2abddd1f914ed3299db3e7a80/11-Figure6-1.png)

Analytical IK solutions typically assume that the end effector's pose is given relative to the robot's base. However, in many practical scenarios, the end effector's pose may be specified relative to a different coordinate system. In our case, all the object's poses are given relative to MuJoCo's world coordinates.

These different coordinate systems are referred to as **reference frames**. In robotics, it is common to work with multiple reference frames, such as the world frame, robot base frame, and end effector frame, and perhaps others depending on the task at hand.

A reference frame is defined by the position of its origin and the orientation of its axes in space, relative to some origin/world frame. The position of the origin is typically represented as a 3D vector $t = (x, y, z)$, called the translation vector. The orientation of the axes is represented using a rotation matrix $R$, which is a 3x3 orthonormal matrix that describes the direction of the frame's axes relative to the world frame.

This rotation matrix transforms coordinates from the local frame to the world frame. That is, given a 3D point $p_A = (x, y, z)$ in space relative to reference frame A, with rotation matrix $R_A$ and translation vector $t_A$, we can convert this point to world coordinates using the following equation:
$$
p_{world} = R_A p_A + t_A = \begin{bmatrix} R_A & t_A \\ 0 & 1 \end{bmatrix} \begin{bmatrix} p_A \\ 1 \end{bmatrix} = T_A p_A
$$
Here, $T_A$ is called the **reference frame transformation**, and it is a canonical representation of 3D pose (position + orientation) in the world frame.

To go the opposite direction, from world coordinates to reference frame A, we simply invert the process:
$$
p_A = R_A^{T} (p_{world} - t_A) = R_A^{T} p_{world} - R_A^{T} t_A = \begin{bmatrix} R_A^{T} & -R_A^{T} t_A \\ 0 & 1 \end{bmatrix} \begin{bmatrix} p_{world} \\ 1 \end{bmatrix} = T_A^{-1} p_{world}
$$

Similarly, to convert an entire pose (position + orientation) $T_B$ to reference frame A, we can use the following equation:
$$
T_A^B = T_A^{-1} T_B = \begin{bmatrix} R_A^{T} R_B & R_A^{T} (t_B - t_A) \\ 0 & 1 \end{bmatrix}
$$

Write a function that transforms a target pose (given as a 3D point and a rotation matrix) from world coordinates to the UR5e base frame coordinates.

**TODO:** implement `transform_world_to_ur5e_base` in `task1.py`.

In [ ]:
# sanity check: transform_world_to_ur5e_base
check_data = get_check_data()
base_pos, base_rot_mat = t1.transform_world_to_ur5e_base(check_data, *t1.get_ee_pose(check_data))

# the robot base is at [0, 0, 0.72] and rotated by -90 degrees around the z axis, so the
# end-effector pose at the home configuration looks different from the base frame
np.testing.assert_allclose(base_pos, [-0.134, 0.492, 0.488], atol=1e-3)
np.testing.assert_allclose(base_rot_mat, np.diag([1, -1, -1]), atol=1e-3)

print(
    "transform_world_to_ur5e_base sanity checks passed. "
    f"end-effector position in the base frame = {np.round(base_pos, 3)}"
)

### Computing Inverse Kinematics Analytically

The `ur-analytic-ik` library provides an analytical solution for the inverse kinematics of the UR5e robot natively in python.

We will now implement a function to compute the inverse kinematics using this library:

**TODO:** implement `compute_ik` in `task1.py`.

Let's test it out. We will bring the robot to it's home configuration and get the end effector's pose. Then, we will use our `compute_ik` function to compute the joint angles that would bring the end effector to this pose and compare the results.

In [ ]:
# Reset state and time to the home keyframe
mujoco.mj_resetDataKeyframe(model, data, model.key("robot_home").id)

# update the Mujoco data structure to reflect the reset state
mujoco.mj_forward(model, data)

# get the end-effector pose at the home configuration (forward kinematics)
home_config_ee_pos, home_config_ee_rot_mat = t1.get_ee_pose(data)

# compute IK solutions for the home configuration end-effector pose
sols = t1.compute_ik(data, home_config_ee_pos, home_config_ee_rot_mat)
print(f"found {len(sols)} IK solutions")

There are multiple solutions. Are all of them valid? Does any of them match the home configuration? To test this, let's visualize each solution in the simulator. First, we start by finding the robot-specific indices for setting the `qpos` value in the data object:

**TODO:** implement `get_entity_qpos_idx` in `task1.py`.

In [ ]:
# sanity check: get_entity_qpos_idx
robot_qpos_ids = list(map(int, t1.get_entity_qpos_idx(model, "robot")))
box_qpos_ids = list(map(int, t1.get_entity_qpos_idx(model, "red_box")))

# each of the robot's six hinge joints has one entry. the box has a free joint with seven
# entries: a 3D position and a quaternion
test.assertEqual(robot_qpos_ids, [35, 36, 37, 38, 39, 40])
test.assertEqual(box_qpos_ids, [0, 1, 2, 3, 4, 5, 6])

print(f"get_entity_qpos_idx sanity checks passed. robot = {robot_qpos_ids}, box = {box_qpos_ids}")

Now let's visualize each IK solution one by one:

In [ ]:
from ipywidgets import interact

# list camera options
CAMERA_LIST = [
    "leftsideview",
    "rightsideview",
    "frontview",
    "birdview",
    "leftangleview",
    "rightangleview",
]

# get the qpos indices corresponding to the robot joints
robot_qpos_idx = t1.get_entity_qpos_idx(model, "robot")


# interactive visualization of IK solutions
@interact(
    ik_solution_number=(0, len(sols) - 1),
    camera=CAMERA_LIST,
)
def render_ik_solution(ik_solution_number, camera="leftsideview"):
    data.qpos[robot_qpos_idx] = sols[
        ik_solution_number
    ]  # apply one of the IK solutions
    mujoco.mj_forward(model, data)  # Update the simulation state
    with mujoco.Renderer(model) as renderer:
        renderer.update_scene(data, camera=camera)
        media.show_image(renderer.render())

Let's repeat the above process to find a pre-grasp configuration, i.e., positioning the end-effector for grasping the red box from above:

**TODO:** implement `get_above_block_pose` in `task1.py`.

In [ ]:
# set the target position and orientation above the box
above_block_pos, above_block_rot_mat = t1.get_above_block_pose(data)

# update the simulation state.
# this is required to ensure that the data structure reflects the reset state
# before calling the IK for the first time.
mujoco.mj_forward(model, data)

# compute IK solutions for the above-box end-effector pose
box_ik_solutions = t1.compute_ik(data, above_block_pos, above_block_rot_mat)

# interactive visualization of IK solutions for above-box pose
@interact(ik_solution_number=(0, len(box_ik_solutions) - 1), camera=CAMERA_LIST)
def render_ik_solution(ik_solution_number, camera="leftsideview"):
    data.qpos[t1.get_entity_qpos_idx(model, "robot")] = box_ik_solutions[
        ik_solution_number
    ]  # apply one of the IK solutions
    mujoco.mj_forward(model, data)  # Update the simulation state
    with mujoco.Renderer(model) as renderer:
        renderer.update_scene(data, camera=camera)
        media.show_image(renderer.render())

Let's select a solution with no collisions and implement a controller to bring the robot to this configuration.

In [ ]:
above_box_config = box_ik_solutions[0]  # select one solution with no collisions

**Next:** continue to Task 2 in `task2.ipynb`.